<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 18 · Logging, Failure Management, and Resilience

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook demonstrates the logging and retry helpers from `ch18_logging_failure_management.py`.

- Configure a rotating file logger and emit a few messages.
- Use `safe_call` to catch and log exceptions from arbitrary functions.
- Apply the `retry_with_backoff` decorator to a tiny demo job.


In [ ]:
import sys
from pathlib import Path
import subprocess
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/yhilpisch/paatcode.git"
    ROOT_DIR = Path("/content/paatcode")
    if not ROOT_DIR.exists():
        subprocess.run([
            "git",
            "clone",
            REPO_URL,
            str(ROOT_DIR),
        ], check=True)
    PROJECT_ROOT = ROOT_DIR
else:
    PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch18_logging_failure_management as ch18  # logging helpers


### 1. Configure Logging

We point the logger at a local file in the `logs/` directory and write a couple of test messages.

In [ ]:
cfg = ch18.LoggingConfig(path=PROJECT_ROOT / "logs" / "chapter18_demo.log")
logger = ch18.configure_logging(cfg)
logger.info("chapter 18 logging demo started")
logger.debug("this debug message is mainly for illustration")
cfg.path


### 2. Safe Calls and Retry Wrapper

Next we define a small function that sometimes fails and run it through `safe_call` and the retry decorator to see how failures are logged.

In [ ]:
def may_fail(x: int) -> float:
    if x % 2 == 0:
        raise ValueError("simulated failure for even x")
    return float(x)

print("Safe call with x = 2:")
print(ch18.safe_call(may_fail, 2))

print("\nSafe call with x = 3:")
print(ch18.safe_call(may_fail, 3))


In [ ]:
@ch18.retry_with_backoff(max_attempts=3, initial_delay=0.2, factor=2.0)
def flaky_once() -> str:
    if not hasattr(flaky_once, "_called"):
        flaky_once._called = True  # type: ignore[attr-defined]
        raise RuntimeError("simulated transient error")
    return "ok after first retry"

print("\nRetry demo:")
print(ch18.safe_call(flaky_once))


### Takeaways

- Lightweight logging and retry helpers go a long way towards making small trading systems resilient.
- Centralising configuration (such as log paths and rotation limits) keeps jobs consistent across machines.
- The same patterns can wrap data collectors, strategy loops, and reporting jobs with only minor adjustments.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">